# Train the Nimbus adapter with QLoRA on a free GPU

Companion notebook for **llm-finetune-lab** by Asad Aslam. Works on Google Colab and Kaggle.

It runs the real pipeline from the repo, so training here is identical to training on
your own machine. Nothing is copied into the notebook that could drift out of sync.

**Before you run anything, turn the GPU on:**

- Colab: Runtime > Change runtime type > T4 GPU
- Kaggle: right sidebar > Accelerator > GPU T4 x2 (and switch Internet on)

QLoRA needs a CUDA GPU. On CPU the pipeline falls back to plain LoRA and tells you so.

## 1. Get the code

In [1]:
REPO = 'https://github.com/asadaslam556/llm-finetune-lab.git'  # change to your fork if needed

!git clone -q {REPO} lab
%cd lab
!pip install -q -e ".[train,quant]"

/content/lab
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.0 MB/s eta 0:00:00
  Building editable for llm-finetune-lab (pyproject.toml) ... done


## 2. Optional: Hugging Face token

Only needed for gated models like Llama or Gemma. The default Qwen model is public.
On Colab, add a secret called `HF_TOKEN` with the key icon in the left sidebar.

In [2]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab secrets')
except Exception:
    print('no HF token, fine for public models')

no HF token, fine for public models


## 3. Pick a model and check the plan

The plan shows whether this GPU will run true 4-bit QLoRA, and the estimated VRAM.
A T4 has no bf16, so expect `compute_dtype: float16`, which is fine.

The default here is `Qwen/Qwen2.5-1.5B-Instruct` (Apache-2.0). `Qwen/Qwen2.5-7B-Instruct` also fits a T4 in 4-bit, but merging it later needs about 15 GB of RAM on your own machine. Avoid Qwen2.5-3B: its licence is non-commercial.

In [3]:
os.environ['LFL_BASE_MODEL_HF'] = 'Qwen/Qwen2.5-1.5B-Instruct'  # Apache-2.0; avoid 3B, it is research-licensed
os.environ['LFL_NUM_EPOCHS'] = '3'   # tiny dataset, a few passes help

!finetune-lab plan

{
  "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
  "hardware": {
    "torch": "2.11.0+cu128",
    "cuda": true,
    "gpu": "Tesla T4",
    "vram_gb": 14.6,
    "bf16": true,
    "bitsandbytes": "0.50.2"
  },
  "plan": {
    "strategy": "qlora",
    "requested": "qlora",
    "bits": 4,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "bfloat16",
    "gradient_checkpointing": true,
    "optimizer": "paged_adamw_8bit",
    "reason": "4-bit nf4 on Tesla T4 via bitsandbytes 0.50.2, compute in bfloat16.",
    "downgraded": false
  },
  "vram_estimate": {
    "params_b": 1.5,
    "qlora_gb": 2.2,
    "lora_bf16_gb": 4.3,
    "saving_gb": 2.1,
    "note": "Estimate only. Activations vary with batch size and sequence length."
  }
}


## 4. Train

Ingest, prepare, pull, profile, fine-tune and evaluate. Export happens on your machine, where Ollama lives.

In [4]:
!finetune-lab run --real --stages ingest prepare pull_base profile finetune evaluate

Run 20260924-210938-50d305 (real) in /content/lab/artifacts/20260924-210938-50d305
  [ok  ] Ingest data: Ingested 137 tickets from 1 file(s), dropped 3.
  [ok  ] Prepare & format: Prepared 112 train / 12 val rows (dropped 8 dupes, 5 too-short).
21:09:38 INFO    httpx :: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
21:09:38 INFO    httpx :: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/revision/main "HTTP/1.1 200 OK"
21:09:38 INFO    httpx :: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/989aa7980e4cf806f80c7fef2b1adb7bc71aa306?recursive=true&expand=false "HTTP/1.1 200 OK"
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0% 0/10 [00:00<?, ?it/s]21:09:39 INFO    httpx :: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/merges.txt "HTTP/1.1 307 Temporary Redirect"
21

## 5. Look at the results

In [6]:
import json, glob
run = sorted(glob.glob('artifacts/2*'))[-1]
info = json.load(open(f'{run}/adapter/ADAPTER_INFO.json'))
print('strategy:', info['config']['strategy'], '| final loss:', info['final_loss'],
      '| trainable:', info.get('trainable_pct'), '%')
report = json.load(open(f'{run}/eval_report.json'))
print('overlap F1:', report['overlap_f1'], '| keyword hit rate:', report['keyword_hit_rate'])
for ex in report['examples'][:3]:
    print('Q:', ex['question'], 'A:', ex['prediction'])

FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/20260924-210938-50d305/adapter/ADAPTER_INFO.json'

## 6. Download the adapter

**Do this before you close the tab.** Notebook disks are wiped when the session ends. The adapter is a few MB, not the whole model.

In [ ]:
import shutil
shutil.make_archive('nimbus-adapter', 'zip', f'{run}/adapter')
try:
    from google.colab import files
    files.download('nimbus-adapter.zip')
except Exception:
    print('download nimbus-adapter.zip from the file browser')

## 7. Back on your machine

1. Do one dry run so `artifacts/<run-id>/` exists: `finetune-lab run`
2. Unzip `nimbus-adapter.zip` into `artifacts/<run-id>/adapter/`
3. Export and deploy for real:

```bash
finetune-lab run --real --stages export_deploy
ollama run nimbus-support
```

The export stage merges the adapter into a full-precision base before converting to GGUF.
Merging into the 4-bit base would lose the adapter, see docs/qlora.md.